# Judge evaluation of LLMS responses

This part was ran in Google Colab
STAGE 1 - Extraction of canonical answer

## Imports and file paths

In [ ]:
import json
import re
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# File paths
merge_results = "merged_results_labels.jsonl"

print("Paths set.")

Paths set.


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving merged_results_labels.jsonl to merged_results_labels (1).jsonl


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Load data

In [ ]:
q_info = pd.read_json(merge_results, lines=True)

## JUDGE MODEL

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "google/gemma-2-9b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## CHAT GENERATION

In [ ]:
def generate_chat(messages, model, tokenizer, max_new_tokens=512):
    """messages: list of {'role','content'} dicts. Returns ONLY new tokens."""
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_tokens = outputs[0][prompt_len:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

print("generate_chat ready.")

generate_chat ready.


## STAGE 1 — Canonical answer extraction

In [ ]:
def extract_canonical_answer(a_list, qtype):
    prompt = f"""
You are extracting the final answer from multiple human answers.

Question type: {qtype}

Human answers:
{a_list}

Instructions:
1. Do NOT solve the problem yourself.
2. Only summarize ideas that human answers already say.
3. Your output should contain:
   - a short summary of the main human ideas that are key to the solution IF such ideas exist
   - the final answer
4. The final answer must follow the correct format depending on question type:

- numeric → final numeric/algebraic expression; include a summary ONLY if humans provided reasoning
- yesno → summarize the key argument + final yes/no
- proof → summarize the key logical steps in 2–5 sentences (do NOT just write "proof")
- concept → summarize the key concept or definition
- open → summarize the main conclusion or classification

7. Output ONLY the summary (if any) and the final answer.
"""
    messages = [{"role": "user", "content": prompt}]
    return generate_chat(messages, model, tokenizer)


In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/NLP_extractions/canonical_checkpoint.jsonl"
FINAL_PATH = "/content/drive/MyDrive/NLP_extractions/canonical_answers.jsonl"

In [ ]:
import os

# Ensure column exists
if "canonical_answer" not in q_info.columns:
    q_info["canonical_answer"] = None

# Resume from checkpoint if available
if os.path.exists(CHECKPOINT_PATH):
    print("Resuming from checkpoint...")
    saved = pd.read_json(CHECKPOINT_PATH, lines=True)
    # align by index
    q_info.loc[saved.index, "canonical_answer"] = saved["canonical_answer"]
else:
    print("No checkpoint found. Starting fresh.")


No checkpoint found. Starting fresh.


In [ ]:
from tqdm import tqdm

for idx, row in tqdm(q_info.iterrows(), total=len(q_info)):
    if pd.notna(row["canonical_answer"]):
        continue  # already done

    can_answer = extract_canonical_answer(row["A_list"], row["question_type"])

    q_info.at[idx, "canonical_answer"] = can_answer

    # save after each row
    q_info.to_json(CHECKPOINT_PATH, orient="records", lines=True)

# final save
q_info.to_json(FINAL_PATH, orient="records", lines=True)
print("Done.")


100%|██████████| 100/100 [10:50<00:00,  6.51s/it]

Done.


In [ ]:
extractions = pd.read_json(FINAL_PATH, lines="True")

In [ ]:
for idx, row in extractions.head(30).iterrows():
    print(f"{idx}: {row['canonical_answer']}\n")

0: Humans found that $b=9$ is a root of the equation $6b^2+24b+27=b^3$ and that $11\times 14\times 15=2310$ which when converted to base $9$ is represented as $3146$. 


$$44$$

1: The main idea is to use a combination of combinatorial techniques and generating functions to solve the problem.  

$$\\binom{19}{3}-\\binom{14}{3}-\\binom{12}{3}-\\binom{13}{3}-\\binom{12}{3}+\\binom{7}{3}+\\binom{6}{3}+\\binom{6}{3}+\\binom{8}{3}+\\binom{7}{3}+\\binom{5}{3}$$

2: Humans suggest changing the variable to $y = x^{1/3}$ to get a quadratic equation. They then calculate the discriminant of this quadratic. 

final answer:  $ \frac{5}{3}y^{2} - \frac{4k}{3}y + k^{2} > 0$

3: The proof strategy involves considering the extremal values of  $w^3$  which is obtained when the line $Y=w^3$ touches the graph of $f(X)=X^3-3uX^2+3v^2X$.  This happens when two variables are equal.

The proof is then carried out in two cases:

* $w^3=0$ (which implies $z=0$)
* $y=z$ 

Final answer:  The inequality holds.

4:

In [ ]:
row = q_info.iloc[0]
can_answer = extract_canonical_answer(row["A_list"], row["question_type"])
print(can_answer)


Humans found that $b=9$ is a root of the equation $6b^2+24b+27=b^3$ and that $11\times 14\times 15=2310$ which when converted to base $9$ is represented as $3146$. 


$$44$$
